# 実習2：乳房腫瘍の良性・悪性を分類しよう

**目標：正答率だけでなく、間違い方を説明する。**

Google Colabでこのファイルを開き、上から順に実行してください。CPUで実行できます。ローカルでは `python -m pip install pandas scikit-learn` で準備します。

データは乳房腫瘤から採取した細胞の画像から計算した特徴量です。画像を直接入力する実習ではありません。教育用の分類実習であり、実際の検診・診断には使いません。

## 1 ライブラリを読み込む

ペンギン実習と同じ決定木を使います。今回は混同行列・再現率・適合率も計算します。

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import recall_score, precision_score


## 2 データを読み込み、ラベルを確認する

scikit-learnに付属するデータなので、CSVのダウンロードは不要です。0が悪性、1が良性です。数字の大小に意味はありません。

In [ ]:
data = load_breast_cancer(as_frame=True)
print(data.target_names)
print("0 = 悪性、1 = 良性")
print("データの形:", data.data.shape)
print(data.data.head())


## 3 入力 X と正解 y を分ける

Xは細胞核の形状などを表す30個の数値、yは良性・悪性の正解ラベルです。欠損の有無と件数も確認します。

In [ ]:
X = data.data
y = data.target
print("特徴量の数:", X.shape[1])
print("欠損値の数:", X.isnull().sum().sum())
print(y.map({0: "悪性", 1: "良性"}).value_counts())


## 4 学習用とテスト用に分割する

約80%で学習し、約20%で評価します。stratify=yは良性・悪性の比率をできるだけ保ちます。random_stateは再現性のために固定します。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("学習用:", len(X_train))
print("テスト用:", len(X_test))


## 5 決定木で学習する

fit()には学習用の入力と正解を渡します。木の深さはペンギン実習と同じ3に固定します。

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
print("学習が終わりました")


## 6 予測し、正答率を求める

predict()に渡すのはX_testだけです。予測後に正解y_testと比較します。

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("正答率:", round(accuracy * 100, 1), "%")


## 7 混同行列で間違い方を確認する

行が実際のクラス、列が予測したクラスです。labels=[0, 1]で悪性・良性の順を指定します。この実習では悪性を「陽性（positive）」として評価します。良性（benign）と陽性（positive）は別の言葉です。

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
cm_table = pd.DataFrame(
    cm,
    index=["実際：悪性", "実際：良性"],
    columns=["予測：悪性", "予測：良性"]
)
print(cm_table)
print("悪性を良性と予測（見逃し）:", cm[0, 1])
print("良性を悪性と予測:", cm[1, 0])


## 8 悪性の再現率と適合率を求める

再現率：実際に悪性のうち、悪性と予測できた割合。適合率：悪性と予測したうち、実際に悪性だった割合。元データは悪性=0なので、必ずpos_label=0を指定します。

In [ ]:
recall = recall_score(y_test, y_pred, pos_label=0)
precision = precision_score(y_test, y_pred, pos_label=0)
print("悪性の再現率:", round(recall * 100, 1), "%")
print("悪性の適合率:", round(precision * 100, 1), "%")


## 9 見逃したデータを取り出す

& は2つの条件が両方成立することを表します。それぞれの条件を括弧で囲みます。表示される番号は元データの行番号です。

In [ ]:
result = pd.DataFrame({"実際": y_test, "予測": y_pred})
missed = result[(result["実際"] == 0) & (result["予測"] == 1)]
print("悪性を良性と予測した行:")
print(missed)


## 10 全件を良性とする予測と比較する

学習しない単純な比較対象です。「全件を良性」と予測したら、正答率と悪性の再現率はどうなるでしょうか。

In [ ]:
all_benign = [1] * len(y_test)
print("全件を良性とした正答率:",
      round(accuracy_score(y_test, all_benign) * 100, 1), "%")
print("全件を良性とした悪性の再現率:",
      round(recall_score(y_test, all_benign, pos_label=0) * 100, 1), "%")


## 確認問題

1. 0と1は、それぞれ何を表しますか。
2. 混同行列のどの場所が「悪性の見逃し」ですか。件数は？
3. 再現率と適合率は、分母がどう違いますか。
4. 全件を良性と予測しても正答率が0%にならないのはなぜですか。
5. この結果だけで実際の検診に使えると言えますか。

数値は自分の実行結果を記録してください。同じテスト結果を見ながら設定を選ぶと公平な評価ができなくなります。設定の比較には別の検証用データなどが必要です。

## データと参考資料

- [scikit-learn：load_breast_cancer](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)
- [UCI：Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)
- [混同行列](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [再現率](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
- [適合率](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)

作成時の実行環境：scikit-learn 1.8.0。ライブラリの版や分割設定により結果が変わる場合があります。